# Detection and recovery pipeline for the synthetic process

This mirrors TEP notebook 2.2 for the six-variable causalAI process: detection scores only the three endogenous process variables, using the fixed validation-selected settings, then a sample-by-sample model-predictive intervention adjusts the three manipulated inputs (`feed_flow`, `feed_concentration`, and `coolant_flow`). It reconstructs a healthy trajectory, predicts a healthy horizon, solves the same quadratic control objective, simulates the controlled process, and tests for five consecutive healthy moving scores. Recovery is measured as the controlled process-state deviation from its healthy reference, not merely as a drop in model residuals. The healthy-trained regression models and lag-2 process/input matrices are used throughout.

As in TEP's recovery phase, post-alarm trajectories are simulated using the healthy-trained process model and residual noise estimated from healthy training data. This tests whether the controller can bring a modeled process back to its healthy reference; it does not demonstrate recovery while a physical fault remains active.

In [1]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Project root: <repo>


In [2]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from carl.causalai_process import ENDOGENOUS_NAMES, MANIPULATED_NAMES, VARIABLE_NAMES
from carl.tep_utils import initialize_fig, update_fig


DATA_DIR = PROJECT_ROOT / "data" / "interim" / "causalAIProcess"
SOURCE_DIR = PROJECT_ROOT / "data" / "processed" / "causalAIProcess"
MODELS_DIR = PROJECT_ROOT / "models" / "causalAIProcess" / "max_lag_selection"
RESULTS_DIR = PROJECT_ROOT / "reports" / "results" / "causalAIProcess"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

alarm_results_path = RESULTS_DIR / "window_and_consecutive_alarm_selection.csv"
thresholds_path = RESULTS_DIR / "alarm_thresholds_by_window.csv"
selected_lag_path = MODELS_DIR / "selected_max_lag.csv"
required_paths = [
    DATA_DIR / "traces_scaled.parquet",
    SOURCE_DIR / "runs.csv",
    SOURCE_DIR / "causal_graph.csv",
    alarm_results_path,
    thresholds_path,
    selected_lag_path,
    MODELS_DIR / "sample_precision_train.npy",
]
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        "Run notebooks 0.0, 0.2, 1.0, and 1.1 first. Missing: "
        + ", ".join(str(path) for path in missing_paths)
    )

df_scaled = pd.read_parquet(DATA_DIR / "traces_scaled.parquet")
runs = pd.read_csv(SOURCE_DIR / "runs.csv")
causal_graph = pd.read_csv(SOURCE_DIR / "causal_graph.csv")
sample_precision = np.load(MODELS_DIR / "sample_precision_train.npy")

max_lag = int(pd.read_csv(selected_lag_path).loc[0, "best_max_lag"])
if max_lag != 2:
    raise ValueError(f"This pipeline constructs lag-2 matrices; selected max lag is {max_lag}.")
alarm_results = pd.read_csv(alarm_results_path)
best_f1 = alarm_results["Trajectory-Phase F1"].max()
best_candidates = alarm_results.loc[alarm_results["Trajectory-Phase F1"].eq(best_f1)]
best_candidates = best_candidates.dropna(subset=["Median Anomaly Detection Delay"])
if best_candidates.empty:
    raise ValueError("No best-F1 configuration has a finite detection delay.")
best_settings = best_candidates.loc[
    best_candidates["Median Anomaly Detection Delay"].idxmin()
]
window = int(best_settings["Window Size"])
consecutive_alarms_threshold = int(best_settings["Consecutive Alarms"])
threshold_table = pd.read_csv(thresholds_path).set_index("Window Size")
anomaly_threshold = float(threshold_table.loc[window, "Threshold"])
if not np.isclose(anomaly_threshold, best_settings["Threshold"]):
    raise ValueError("Saved threshold does not match the validation-selected configuration.")

manipulated = list(MANIPULATED_NAMES)
process_children = list(ENDOGENOUS_NAMES)
score_children = list(process_children)
feature_names = list(VARIABLE_NAMES)
child_parents_pairs = {
    child: sorted(causal_graph.loc[causal_graph["target"].eq(child), "source"].unique())
    for child in feature_names
}
model_features = {
    child: list(dict.fromkeys(parents + [child]))
    for child, parents in child_parents_pairs.items()
}
variable_idx = {name: idx for idx, name in enumerate(feature_names)}
process_idx = [variable_idx[name] for name in process_children]
manipulated_idx = [variable_idx[name] for name in manipulated]
test_manifest = runs.loc[
    runs["split"].eq("test") & runs["fault_type"].ne("healthy")
].sort_values(["fault_type", "run_id"])
selected_runs = test_manifest.set_index("run_id")
if selected_runs.empty:
    raise ValueError("No faulty test runs are available for the pipeline.")

print(f"Lag={max_lag}; window={window}; consecutive alarms={consecutive_alarms_threshold}")
print(f"Validation-selected threshold={anomaly_threshold:.6g}")
print(f"Faulty held-out runs processed by the counterfactual pipeline: {len(selected_runs)}")

Lag=2; window=8; consecutive alarms=5
Validation-selected threshold=2.97599
Faulty held-out runs processed by the counterfactual pipeline: 225


In [3]:
beta_by_child = {}
a_lag = {lag: np.zeros((len(process_children), len(process_children))) for lag in (1, 2)}
u_lag = {lag: np.zeros((len(process_children), len(manipulated))) for lag in (1, 2)}

for child in feature_names:
    beta_path = MODELS_DIR / f"beta_child_{child}_lag_{max_lag}.npy"
    if not beta_path.is_file():
        raise FileNotFoundError(f"Missing healthy-trained model: {beta_path}")
    beta = np.load(beta_path)
    names = model_features[child]
    expected_size = 1 + max_lag * len(names)
    if beta.size != expected_size:
        raise ValueError(f"Unexpected coefficient count for {child}: {beta.size} != {expected_size}")
    beta_by_child[child] = beta

for process_row, child in enumerate(process_children):
    beta = beta_by_child[child]
    names = model_features[child]
    for lag in (1, 2):
        start = 1 + (lag - 1) * len(names)
        lag_coefficients = beta[start : start + len(names)]
        for name, coefficient in zip(names, lag_coefficients, strict=True):
            if name in process_children:
                a_lag[lag][process_row, process_children.index(name)] = coefficient
            if name in manipulated:
                u_lag[lag][process_row, manipulated.index(name)] = coefficient

if score_children != process_children:
    raise ValueError("Detection and recovery must use the endogenous process variables.")
if sample_precision.shape != (len(score_children), len(score_children)):
    raise ValueError("Residual precision matrix does not match the scored-variable count.")
sample_covariance = np.linalg.pinv(sample_precision)
process_covariance = sample_covariance
process_precision = np.linalg.pinv(process_covariance)

horizon = 10
lambda_w = 1.0
consecutive_healthy_threshold = consecutive_alarms_threshold
quantile_recovery_threshold = 0.95
response = [np.zeros_like(u_lag[1]) for _ in range(horizon)]
if horizon > 1:
    response[1] = u_lag[1].copy()
if horizon > 2:
    response[2] = a_lag[1] @ response[1] + u_lag[2]
for delay in range(3, horizon):
    response[delay] = a_lag[1] @ response[delay - 1] + a_lag[2] @ response[delay - 2]

g_matrix = np.zeros(
    (horizon * len(process_children), (horizon - 1) * len(manipulated))
)
for output_step in range(horizon):
    for control_step in range(min(output_step + 1, horizon - 1)):
        row = slice(output_step * len(process_children), (output_step + 1) * len(process_children))
        column = slice(control_step * len(manipulated), (control_step + 1) * len(manipulated))
        g_matrix[row, column] = response[output_step - control_step]

difference_matrix = np.zeros(
    ((horizon - 2) * len(manipulated), (horizon - 1) * len(manipulated))
)
identity_controls = np.eye(len(manipulated))
for step in range(horizon - 2):
    rows = slice(step * len(manipulated), (step + 1) * len(manipulated))
    current = slice(step * len(manipulated), (step + 1) * len(manipulated))
    following = slice((step + 1) * len(manipulated), (step + 2) * len(manipulated))
    difference_matrix[rows, current] = -identity_controls
    difference_matrix[rows, following] = identity_controls
first_control_matrix = np.zeros((len(manipulated), (horizon - 1) * len(manipulated)))
first_control_matrix[:, : len(manipulated)] = identity_controls
control_penalty_matrix = (
    difference_matrix.T @ difference_matrix
    + first_control_matrix.T @ first_control_matrix
)
if g_matrix.shape != (horizon * 3, (horizon - 1) * 3):
    raise ValueError(f"Unexpected lag-2 control influence matrix shape: {g_matrix.shape}")

print(f"A_1/A_2 shapes: {a_lag[1].shape}, {a_lag[2].shape}")
print(f"U_1/U_2 shapes: {u_lag[1].shape}, {u_lag[2].shape}")
print(f"Lag-2 control influence matrix G: {g_matrix.shape}")

A_1/A_2 shapes: (3, 3), (3, 3)
U_1/U_2 shapes: (3, 3), (3, 3)
Lag-2 control influence matrix G: (30, 27)


In [4]:
def linear_process_step(history, controls=None, process_noise=None):
    next_row = np.empty(len(feature_names), dtype=float)
    for child in feature_names:
        if child in manipulated and controls is not None:
            next_row[variable_idx[child]] = controls[manipulated.index(child)]
            continue
        beta = beta_by_child[child]
        names = model_features[child]
        lagged_features = np.concatenate(
            [history[-lag, [variable_idx[name] for name in names]] for lag in (1, 2)]
        )
        next_row[variable_idx[child]] = beta[0] + lagged_features @ beta[1:]
    if process_noise is not None:
        next_row[process_idx] += process_noise
    return next_row


def linear_rollout(history, control_sequence=None):
    state = history.copy()
    predicted = []
    sequences = [None] * horizon if control_sequence is None else control_sequence
    for controls in sequences:
        next_row = linear_process_step(state, controls)
        state = np.vstack([state, next_row])
        predicted.append(next_row)
    return np.asarray(predicted)


def residual_vector(scaled_run, step, children):
    residual = np.empty(len(children), dtype=float)
    for child_index, child in enumerate(children):
        names = model_features[child]
        lagged_features = np.concatenate(
            [scaled_run[step - lag, [variable_idx[name] for name in names]] for lag in (1, 2)]
        )
        predicted = beta_by_child[child][0] + lagged_features @ beta_by_child[child][1:]
        residual[child_index] = scaled_run[step, variable_idx[child]] - predicted
    return residual


def residual_score(scaled_run, step, children, precision):
    residual = residual_vector(scaled_run, step, children)
    return float((residual @ precision @ residual) / len(children))




In [5]:
pipeline_rows = []
score_rows = []
control_rows = []

for run_id, run in selected_runs.iterrows():
    onset = int(run["fault_start_step"])
    scaled_run = (
        df_scaled.loc[df_scaled["run_id"].eq(run_id)]
        .sort_values("time_step")[feature_names]
        .to_numpy()
    )
    if len(scaled_run) <= max_lag + window:
        raise ValueError(f"Run {run_id} has an invalid trajectory length.")

    observed_scores = np.full(len(scaled_run), np.nan)
    observed_recovery_scores = np.full(len(scaled_run), np.nan)
    for step in range(max_lag, len(scaled_run)):
        observed_scores[step] = residual_score(scaled_run, step, score_children, sample_precision)
        observed_recovery_scores[step] = residual_score(
            scaled_run, step, process_children, process_precision
        )
    observed_moving_scores = pd.Series(observed_scores).rolling(
        window=window, min_periods=window
    ).mean().to_numpy()
    observed_recovery_moving_scores = pd.Series(observed_recovery_scores).rolling(
        window=window, min_periods=window
    ).mean().to_numpy()
    raw_breaches = np.isfinite(observed_moving_scores) & (observed_moving_scores > anomaly_threshold)
    observed_alarms = (
        pd.Series(raw_breaches.astype(int))
        .rolling(window=consecutive_alarms_threshold, min_periods=consecutive_alarms_threshold)
        .sum()
        .eq(consecutive_alarms_threshold)
        .to_numpy(dtype=bool)
    )
    alarm_indices = np.flatnonzero(observed_alarms & (np.arange(len(observed_alarms)) >= onset))
    detection_step = int(alarm_indices[0]) if len(alarm_indices) else None
    fault_detected = detection_step is not None and detection_step >= onset
    detection_delay = detection_step - onset if fault_detected else np.nan
    intervention_step = detection_step + 1 if detection_step is not None else None

    recovery_state_scores = np.full(len(scaled_run), np.nan)
    recovery_moving_scores = np.full(len(scaled_run), np.nan)
    recovery_threshold = np.nan
    process_recovered = False
    recovery_steps = np.nan
    if detection_step is not None and intervention_step < len(scaled_run):
        control_effort = 0.0
        consecutive_healthy_states = 0
        estimated_anomaly_start = intervention_step - consecutive_alarms_threshold - window + 1
        calibration_scores = observed_recovery_moving_scores[
            max_lag + window - 1 : estimated_anomaly_start
        ]
        calibration_scores = calibration_scores[np.isfinite(calibration_scores)]
        if not len(calibration_scores):
            raise ValueError(f"Cannot estimate a recovery threshold for run {run_id}.")
        recovery_threshold = float(
            np.quantile(calibration_scores, quantile_recovery_threshold)
        )

        controlled_ts = np.full_like(scaled_run, np.nan)
        controlled_ts[:intervention_step] = scaled_run[:intervention_step]
        healthy_ts = np.full_like(scaled_run, np.nan)
        history_start = estimated_anomaly_start - max_lag
        healthy_ts[history_start:estimated_anomaly_start] = scaled_run[
            history_start:estimated_anomaly_start
        ]
        for step in range(estimated_anomaly_start, intervention_step):
            healthy_ts[step] = linear_process_step(
                healthy_ts[step - max_lag : step]
            )

        rng = np.random.default_rng(0)
        for step in range(intervention_step, len(scaled_run)):
            history = controlled_ts[step - max_lag : step]
            healthy_history = healthy_ts[step - max_lag : step]
            healthy_hat = linear_rollout(healthy_history)
            current_tilde = linear_rollout(
                history,
                np.zeros((horizon, len(manipulated))),
            )
            process_child_diff = (
                current_tilde[:, process_idx] - healthy_hat[:, process_idx]
            ).reshape(-1)
            previous_controls = controlled_ts[step - 1, manipulated_idx]
            q_matrix = 2 * (
                g_matrix.T @ g_matrix + lambda_w * control_penalty_matrix
            )
            c_vector = 2 * (
                g_matrix.T @ process_child_diff
                - lambda_w * first_control_matrix.T @ previous_controls
            )
            control_sequence = np.linalg.solve(q_matrix, -c_vector)
            action = control_sequence[: len(manipulated)]
            control_change = action - previous_controls
            control_effort += float(control_change @ control_change)

            process_noise = rng.multivariate_normal(
                mean=np.zeros(len(process_children)),
                cov=process_covariance,
            )
            controlled_ts[step] = linear_process_step(
                history,
                controls=action,
                process_noise=process_noise,
            )
            healthy_ts[step] = linear_process_step(healthy_history)
            state_error = controlled_ts[step, process_idx] - healthy_ts[step, process_idx]
            recovery_state_scores[step] = float(
                (state_error @ process_precision @ state_error) / len(process_children)
            )

            if step >= intervention_step + window - 1:
                recovery_moving_scores[step] = float(
                    np.mean(recovery_state_scores[step - window + 1 : step + 1])
                )
                if recovery_moving_scores[step] < recovery_threshold:
                    consecutive_healthy_states += 1
                else:
                    consecutive_healthy_states = 0
                if consecutive_healthy_states == consecutive_healthy_threshold:
                    process_recovered = True
                    recovery_steps = step - intervention_step + 1

            control_rows.append(
                {
                    "run_id": run_id,
                    "time_step": step,
                    **{name: float(action[idx]) for idx, name in enumerate(manipulated)},
                    "recovery_score": recovery_state_scores[step],
                    "recovery_moving_score": recovery_moving_scores[step],
                    **{
                        f"{name}_state_error": float(state_error[idx])
                        for idx, name in enumerate(process_children)
                    },
                    **{
                        f"{name}_controlled": float(controlled_ts[step, variable_idx[name]])
                        for name in process_children
                    },
                    **{
                        f"{name}_healthy_reference": float(healthy_ts[step, variable_idx[name]])
                        for name in process_children
                    },
                    **{
                        f"{name}_healthy_reference": float(healthy_ts[step, variable_idx[name]])
                        for name in manipulated
                    },
                }
            )
            if process_recovered:
                break

    pipeline_rows.append(
        {
            "run_id": run_id,
            "fault_type": run["fault_type"],
            "fault_start_step": onset,
            "anomaly_detected": detection_step is not None,
            "fault_detected": fault_detected,
            "detection_step": detection_step,
            "detection_delay_samples": detection_delay,
            "intervention_step": intervention_step,
            "recovery_threshold": recovery_threshold,
            "process_recovered": process_recovered,
            "recovery_steps": recovery_steps,
            "control_effort": control_effort if detection_step is not None and intervention_step < len(scaled_run) else np.nan,
            "window_size": window,
            "consecutive_alarms": consecutive_alarms_threshold,
            "anomaly_threshold": anomaly_threshold,
        }
    )
    score_rows.append(
        pd.DataFrame(
            {
                "run_id": run_id,
                "time_step": np.arange(len(scaled_run)),
                "observed_score": observed_scores,
                "observed_moving_score": observed_moving_scores,
                "observed_alarm": observed_alarms.astype(int),
                "observed_process_score": observed_recovery_scores,
                "observed_process_moving_score": observed_recovery_moving_scores,
                "recovery_state_score": recovery_state_scores,
                "recovery_moving_score": recovery_moving_scores,
            }
        )
    )
    print(
        f"{run_id}: fault detected={fault_detected}, delay={detection_delay}, "
        f"recovered={process_recovered}, recovery steps={recovery_steps}"
    )

results = pd.DataFrame(pipeline_rows)
scores = pd.concat(score_rows, ignore_index=True)
controls = pd.DataFrame(control_rows)
results.to_csv(RESULTS_DIR / "detection_recovery_results.csv", index=False)
scores.to_parquet(RESULTS_DIR / "detection_recovery_scores.parquet", index=False)
controls.to_parquet(RESULTS_DIR / "detection_recovery_controls.parquet", index=False)
display(results)


run_hydraulic_coupling_shift_00506: fault detected=False, delay=nan, recovered=False, recovery steps=nan
run_hydraulic_coupling_shift_00512: fault detected=True, delay=323, recovered=True, recovery steps=17
run_hydraulic_coupling_shift_00515: fault detected=True, delay=136, recovered=True, recovery steps=13
run_hydraulic_coupling_shift_00532: fault detected=False, delay=nan, recovered=False, recovery steps=nan
run_hydraulic_coupling_shift_00533: fault detected=True, delay=591, recovered=True, recovery steps=14
run_hydraulic_coupling_shift_00536: fault detected=False, delay=nan, recovered=False, recovery steps=nan
run_hydraulic_coupling_shift_00539: fault detected=True, delay=423, recovered=True, recovery steps=13
run_hydraulic_coupling_shift_00549: fault detected=True, delay=327, recovered=True, recovery steps=16
run_hydraulic_coupling_shift_00554: fault detected=True, delay=666, recovered=True, recovery steps=12
run_hydraulic_coupling_shift_00559: fault detected=True, delay=189, recov

,run_id,fault_type,fault_start_step,anomaly_detected,fault_detected,detection_step,detection_delay_samples,intervention_step,recovery_threshold,process_recovered,recovery_steps,control_effort,window_size,consecutive_alarms,anomaly_threshold
0,run_hydraulic_coupling_shift_00506,hydraulic_coupling_shift,200,False,False,NaN,NaN,NaN,NaN,False,NaN,NaN,8,5,2.97599
1,run_hydraulic_coupling_shift_00512,hydraulic_coupling_shift,200,True,True,523.0,323.0,524.0,1.555912,True,17.0,13.187387,8,5,2.97599
2,run_hydraulic_coupling_shift_00515,hydraulic_coupling_shift,200,True,True,336.0,136.0,337.0,2.457264,True,13.0,1.950117,8,5,2.97599
3,run_hydraulic_coupling_shift_00532,hydraulic_coupling_shift,200,False,False,NaN,NaN,NaN,NaN,False,NaN,NaN,8,5,2.97599
4,run_hydraulic_coupling_shift_00533,hydraulic_coupling_shift,200,True,True,791.0,591.0,792.0,2.480729,True,14.0,10.187132,8,5,2.97599
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220,run_thermal_coupling_shift_01446,thermal_coupling_shift,200,True,True,513.0,313.0,514.0,2.412328,True,13.0,1.057220,8,5,2.97599
221,run_thermal_coupling_shift_01452,thermal_coupling_shift,200,True,True,209.0,9.0,210.0,1.322453,True,14.0,1.263838,8,5,2.97599
222,run_thermal_coupling_shift_01461,thermal_coupling_shift,200,True,True,279.0,79.0,280.0,1.772147,True,15.0,2.494707,8,5,2.97599
223,run_thermal_coupling_shift_01470,thermal_coupling_shift,200,True,True,244.0,44.0,245.0,1.640576,True,14.0,1.992901,8,5,2.97599


# 2. Detection and recovery plots

In [6]:
with_plot = True
legend_loc = "upper right"
xmax = len(scores.loc[scores["run_id"].eq(selected_runs.index[0])]) - 1
xticks = list(range(0, xmax + 1, max(1, xmax // 24)))
y_lims = (1e-2, 1e3)
yticks = [10.0**power for power in range(-2, 4)] + [anomaly_threshold]

for ts_idx, (_, result) in enumerate(results.iterrows()):
    run_id = result["run_id"]
    run_scores = scores.loc[scores["run_id"].eq(run_id)]
    anomaly_scores = run_scores["observed_score"].to_numpy().copy()
    anomaly_moving_averages = run_scores["observed_moving_score"].to_numpy().copy()
    recovery_scores = run_scores["recovery_state_score"].to_numpy().copy()
    recovery_moving_averages = run_scores["recovery_moving_score"].to_numpy().copy()
    anomaly_scores[:max_lag] = np.nan
    anomaly_moving_averages[: max_lag + window - 1] = np.nan
    if result["fault_detected"]:
        detection_step = int(result["detection_step"])
        anomaly_scores[detection_step + 1 :] = np.nan
        anomaly_moving_averages[detection_step + 1 :] = np.nan

    if with_plot:
        fig, ax, [
            recovery_score_line,
            recovery_moving_average_line,
            anomaly_score_line,
            anomaly_moving_average_line,
        ] = initialize_fig(
            xmax,
            xticks,
            y_lims,
            yticks,
            anomaly_threshold,
            legend_loc,
            ts_idx,
        )
        update_fig(
            [anomaly_score_line, anomaly_moving_average_line],
            [anomaly_scores, anomaly_moving_averages],
            xmax,
        )
        update_fig(
            [recovery_score_line, recovery_moving_average_line],
            [recovery_scores, recovery_moving_averages],
            xmax,
        )
        ax.set_title(f"Anomaly and Recovery Scores - {run_id}", fontsize=10)
        ax.vlines(
            result["fault_start_step"],
            ymin=y_lims[0],
            ymax=y_lims[1],
            color="#D62020",
            label="Anomaly",
        )
        if result["fault_detected"]:
            ax.vlines(
                result["detection_step"],
                ymin=y_lims[0],
                ymax=y_lims[1],
                color="#B62A9C",
                label=f"Anomaly Detected (x = {int(result['detection_step'])})",
            )
            ax.hlines(
                result["recovery_threshold"],
                xmin=0.0,
                xmax=xmax,
                colors="#027204",
                label=f"Recovery Threshold (y = {result['recovery_threshold']:.2f})",
            )
            if result["process_recovered"]:
                recovery_step = int(result["intervention_step"] + result["recovery_steps"] - 1)
                ax.vlines(
                    recovery_step,
                    ymin=y_lims[0],
                    ymax=y_lims[1],
                    color="#49AB08",
                    label=f"Recovery Completed (x = {recovery_step})",
                )
        ax.legend(loc=legend_loc)
        display(fig)
        plt.close(fig)


<Figure size 1600x700 with 1 Axes>

<Figure size 1600x700 with 1 Axes>

<Figure size 1600x700 with 1 Axes>

<Figure size 1600x700 with 1 Axes>

<Figure size 1600x700 with 1 Axes>

<Figure size 1600x700 with 1 Axes>

... (215 outputs left out)

<Figure size 1600x700 with 1 Axes>

<Figure size 1600x700 with 1 Axes>

<Figure size 1600x700 with 1 Axes>

<Figure size 1600x700 with 1 Axes>

In [7]:
detection_rate = results["fault_detected"].mean()
detected = results.loc[results["fault_detected"]]
recovery_rate = detected["process_recovered"].mean() if len(detected) else np.nan
median_delay = detected["detection_delay_samples"].median() if len(detected) else np.nan
recovered = detected.loc[detected["process_recovered"]]
median_recovery_steps = recovered["recovery_steps"].median() if len(recovered) else np.nan

print(f"Fault detection rate: {detection_rate:.2%}")
print(f"Median fault-detection delay: {median_delay} samples")
print(f"Recovery rate among detected faults: {recovery_rate:.2%}")
print(f"Median recovery time: {median_recovery_steps} samples")
display(
    results.groupby("fault_type", as_index=False).agg(
        runs=("run_id", "count"),
        fault_detection_rate=("fault_detected", "mean"),
        recovery_rate=("process_recovered", "mean"),
        median_detection_delay=("detection_delay_samples", "median"),
        median_recovery_steps=("recovery_steps", "median"),
    )
)
print(f"Saved results under {RESULTS_DIR}")

Fault detection rate: 83.56%
Median fault-detection delay: 31.5 samples
Recovery rate among detected faults: 100.00%
Median recovery time: 15.0 samples


,fault_type,runs,fault_detection_rate,recovery_rate,median_detection_delay,median_recovery_steps
0,hydraulic_coupling_shift,75,0.546667,0.546667,194.0,14.0
1,reaction_coupling_shift,75,1.000000,1.000000,19.0,15.0
2,thermal_coupling_shift,75,0.960000,0.960000,36.0,14.0


Saved results under <repo>\reports\results\causalAIProcess
